# 4. 4번째 코드

In [ ]:
import requests
import pandas as pd
import time
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse

ACCESS_TOKEN = ""
IG_USER_ID = "" ## 내 아이디
HASHTAG_ID = "" ## 먼작귀 검색 해시태그 이이디

# Avf1NY7r5_ZN4H3pQJ9ws2_
# ADBIc9DLNjcvpVP1pnhIeEl

TARGET_COUNT = 2000
PAGE_LIMIT = 40
MAX_PAGES = 150

url = f"https://graph.facebook.com/v26.0/{HASHTAG_ID}/top_media"

params = {
    "user_id": IG_USER_ID,
    "fields": "id,caption,media_type,permalink,timestamp,like_count,comments_count",
    "limit": PAGE_LIMIT,
    "access_token": ACCESS_TOKEN
}

# =====================================
# 2. 데이터 저장 변수
# =====================================

all_posts = []
seen_ids = set()

page_num = 0
no_new_pages = 0

# =====================================
# 3. 인스타그램 게시글 수집
# =====================================

while len(all_posts) < TARGET_COUNT and page_num < MAX_PAGES:

    try:

        # ---------------------------------
        # API 요청 및 오류 재시도
        # ---------------------------------

        success = False

        for retry in range(3):

            response = requests.get(
                url,
                params=params,
                timeout=60
            )

            result = response.json()

            if response.status_code == 200 and "error" not in result:
                success = True
                break

            print(
                f"API 오류 ({retry + 1}/3):",
                result.get("error", result)
            )

            # 인증/권한 오류는 재시도하지 않음
            error_code = result.get("error", {}).get("code")

            if error_code in [104, 190, 200]:
                print("인증 또는 권한 오류로 중단")
                break

            if retry < 2:
                wait_time = 10 * (2 ** retry)

                print(f"{wait_time}초 후 재시도")
                time.sleep(wait_time)

        if not success:
            print("API 요청 실패로 수집 중단")
            break

        # ---------------------------------
        # 게시글 데이터 처리
        # ---------------------------------

        posts = result.get("data", [])

        page_num += 1
        new_count = 0

        for post in posts:

            post_id = post.get("id")

            if post_id and post_id not in seen_ids:

                seen_ids.add(post_id)
                all_posts.append(post)
                new_count += 1

            if len(all_posts) >= TARGET_COUNT:
                break

        # ---------------------------------
        # 수집 현황 출력
        # ---------------------------------

        print(f"\n{page_num}페이지")
        print(f"응답 건수: {len(posts)}")
        print(f"신규 수집: {new_count}")
        print(f"누적 수집: {len(all_posts)}")

        # ---------------------------------
        # 목표 수집량 확인
        # ---------------------------------

        if len(all_posts) >= TARGET_COUNT:
            print("목표 수집량 달성")
            break

        # ---------------------------------
        # 신규 게시글 여부 확인
        # ---------------------------------

        if new_count == 0:
            no_new_pages += 1
        else:
            no_new_pages = 0

        if no_new_pages >= 2:
            print("신규 게시글이 2페이지 연속 없어 종료")
            break

        # ---------------------------------
        # 다음 페이지 확인
        # ---------------------------------

        paging = result.get("paging", {})
        next_url = paging.get("next")

        if not next_url:
            print("다음 페이지가 없어 종료")
            break

        # ---------------------------------
        # 다음 페이지 URL 및 토큰 유지
        # ---------------------------------

        parsed = urlparse(next_url)
        query_params = parse_qs(parsed.query)

        query_params["access_token"] = [ACCESS_TOKEN]

        url = urlunparse(
            parsed._replace(
                query=urlencode(
                    query_params,
                    doseq=True
                )
            )
        )

        params = None

        # ---------------------------------
        # API 요청 간격
        # ---------------------------------

        time.sleep(2)

    except requests.exceptions.RequestException as e:
        print("네트워크 오류:", e)
        break

    except ValueError as e:
        print("응답 처리 오류:", e)
        break

# =====================================
# 4. CSV 저장
# =====================================

df = pd.DataFrame(
    all_posts,
    columns=[
        "id",
        "caption",
        "media_type",
        "permalink",
        "timestamp",
        "like_count",
        "comments_count"
    ]
)

df.to_csv(
    "instagram_먼작귀2.csv",
    index=False,
    encoding="utf-8-sig"
)

# =====================================
# 5. 최종 결과 출력
# =====================================

print("\n==========================")
print("인스타그램 데이터 수집 종료")
print("==========================")

print(f"목표 수집량: {TARGET_COUNT}건")
print(f"실제 수집량: {len(df)}건")
print(f"요청 페이지: {page_num}개")
print("저장 파일: instagram_먼작귀2.csv")


# 3. 3번째 코드

In [ ]:
import requests
import pandas as pd
import time
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse

# =====================================
# 1. API 설정
# =====================================

ACCESS_TOKEN = "EAAgjgrrdkIUBS34SZA9jqa2UlUIZBaa0qZBduFkm7ZBL7ST3nE79mi4wkxw7iEa6mZAYmozZCPGXx9yvyO4HCeS0rWnPnDkMeC1CLUJkZCdjxemtSO3rbZBy7wcqOYUjm1NrX3UcRZCSS1oPAk2Fg0MGomiEKYvqVeNtJKCjorGNgGawveHZAEbIu5YWxRK6IuNFwG35UhcAZBJU3byBl3l3NewYFNYn3P5MPEh"
IG_USER_ID = "17841480144439476" ## 내 아이디
HASHTAG_ID = "17865460175182032" ## 검색하고자 하는 해시태그 이이디

TARGET_COUNT = 2000
PAGE_LIMIT = 10
MAX_PAGES = 150

url = f"https://graph.facebook.com/v26.0/{HASHTAG_ID}/top_media"

params = {
    "user_id": IG_USER_ID,
    "fields": "id,caption,media_type,permalink,timestamp,like_count,comments_count",
    "limit": PAGE_LIMIT,
    "access_token": ACCESS_TOKEN
}

all_posts = []
seen_ids = set()
page_num = 0
no_new_pages = 0

while len(all_posts) < TARGET_COUNT and page_num < MAX_PAGES:

    try:
        response = requests.get(
            url,
            params=params,
            timeout=60
        )

        result = response.json()

        if response.status_code != 200 or "error" in result:
            print("API 오류:", result.get("error", result))
            break

        posts = result.get("data", [])
        page_num += 1
        new_count = 0

        for post in posts:
            post_id = post.get("id")

            if post_id and post_id not in seen_ids:
                seen_ids.add(post_id)
                all_posts.append(post)
                new_count += 1

            if len(all_posts) >= TARGET_COUNT:
                break

        print(f"\n{page_num}페이지")
        print(f"응답 건수: {len(posts)}")
        print(f"신규 수집: {new_count}")
        print(f"누적 수집: {len(all_posts)}")

        if len(all_posts) >= TARGET_COUNT:
            print("목표 수집량 달성")
            break

        # 신규 데이터가 없는 페이지 확인
        if new_count == 0:
            no_new_pages += 1
        else:
            no_new_pages = 0

        if no_new_pages >= 2:
            print("신규 게시글이 2페이지 연속 없어 종료")
            break

        # 다음 페이지 확인
        paging = result.get("paging", {})
        next_url = paging.get("next")

        if not next_url:
            print("다음 페이지가 없어 종료")
            break

        parsed = urlparse(next_url)
        query_params = parse_qs(parsed.query)

        # 액세스 토큰 유지
        query_params["access_token"] = [ACCESS_TOKEN]

        url = urlunparse(
            parsed._replace(
                query=urlencode(query_params, doseq=True)
            )
        )

        params = None

        time.sleep(2)

    except requests.exceptions.RequestException as e:
        print("요청 오류:", e)
        break

    except ValueError as e:
        print("응답 처리 오류:", e)
        break

# CSV 저장
df = pd.DataFrame(all_posts)

df.to_csv(
    "instagram_치이카와_1027_02.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n====================")
print("인스타그램 수집 종료")
print("====================")
print(f"목표 수집량: {TARGET_COUNT}건")
print(f"실제 수집량: {len(df)}건")
print(f"요청 페이지: {page_num}개")

# 1. 최초 코드

In [17]:
import requests
import pandas as pd

ACCESS_TOKEN="EAAgjgrrdkIUBS7ysfICR8laZBiPFiDKbLHMvXvHeThCjz9QWvkf1idmLORYysbZBO6JDA85LEWZBZAafEOXFasY6dCS6XVs5RIiH516wbxcatmMwTHZBY6SHZAj3cxca72RI9YENJqFaFCzaelmIxmZAuoSNZA84SGP9UCvWTp5tIUkkSaZBqKgg0gkJ9VBFSO2zgYKdUDqVyFGHp6rM2RZBUxYSFdlpQrPeUE"
IG_USER_ID="17841480144439476"
HASHTAG_ID="17865460175182032"

# 노션 8번에서 가장 아래 값
# 17865460175182032/recent_media?user_id=17841480144439476&fields=id,caption,media_type,permailink,timestamp&limit=10

url = f"https://graph.facebook.com/v26.0/{HASHTAG_ID}/top_media"

params = {
    "user_id": IG_USER_ID,
    "fields": "id,caption,media_type,permalink,timestamp,like_count,comments_count",
    "limit": 10,
    "access_token": ACCESS_TOKEN
}

all_posts = list()

while True :
    response = requests.get(url, params=params, timeout=200)
    result = response.json()

    posts = result.get("data", [])

    all_posts.extend(posts)

    print(f"현재까지 수집: {len(all_posts)}건")

    url = result.get("paging", {}).get("next")

    if not url : 
        break
        
    params = None

df = pd.DataFrame(all_posts)

df.to_csv("instagram_치이카와_top.csv", index=False, encoding="utf-8-sig")

print("수집완료")

현재까지 수집: 10건
현재까지 수집: 20건
현재까지 수집: 30건
현재까지 수집: 40건
현재까지 수집: 50건
현재까지 수집: 60건
현재까지 수집: 70건
현재까지 수집: 80건
현재까지 수집: 90건
현재까지 수집: 100건
현재까지 수집: 110건
현재까지 수집: 120건
현재까지 수집: 130건
현재까지 수집: 140건
현재까지 수집: 150건
현재까지 수집: 160건
현재까지 수집: 170건
현재까지 수집: 170건
수집완료


# 2. 두 번째 코드

In [ ]:
import requests
import pandas as pd
import time
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse

# =====================================
# 1. API 설정
# =====================================

ACCESS_TOKEN = "EAAgjgrrdkIUBS7ysfICR8laZBiPFiDKbLHMvXvHeThCjz9QWvkf1idmLORYysbZBO6JDA85LEWZBZAafEOXFasY6dCS6XVs5RIiH516wbxcatmMwTHZBY6SHZAj3cxca72RI9YENJqFaFCzaelmIxmZAuoSNZA84SGP9UCvWTp5tIUkkSaZBqKgg0gkJ9VBFSO2zgYKdUDqVyFGHp6rM2RZBUxYSFdlpQrPeUE"
IG_USER_ID = "17841480144439476" ## 내 아이디
HASHTAG_ID = "17865460175182032" ## 치이카와 검색 해시태그 이이디

TARGET_COUNT = 1000
PAGE_LIMIT = 10
MAX_PAGES = 150

url = f"https://graph.facebook.com/v26.0/{HASHTAG_ID}/top_media"

params = {
    "user_id": IG_USER_ID,
    "fields": "id,caption,media_type,permalink,timestamp,like_count,comments_count",
    "limit": PAGE_LIMIT,
    "access_token": ACCESS_TOKEN
}

all_posts = []
seen_ids = set()
page_num = 0
no_new_pages = 0

while len(all_posts) < TARGET_COUNT and page_num < MAX_PAGES:

    try:
        response = requests.get(
            url,
            params=params,
            timeout=60
        )

        result = response.json()

        if response.status_code != 200 or "error" in result:
            print("API 오류:", result.get("error", result))
            break

        posts = result.get("data", [])
        page_num += 1
        new_count = 0

        for post in posts:
            post_id = post.get("id")

            if post_id and post_id not in seen_ids:
                seen_ids.add(post_id)
                all_posts.append(post)
                new_count += 1

            if len(all_posts) >= TARGET_COUNT:
                break

        print(f"\n{page_num}페이지")
        print(f"응답 건수: {len(posts)}")
        print(f"신규 수집: {new_count}")
        print(f"누적 수집: {len(all_posts)}")

        if len(all_posts) >= TARGET_COUNT:
            print("목표 수집량 달성")
            break

        # 신규 데이터가 없는 페이지 확인
        if new_count == 0:
            no_new_pages += 1
        else:
            no_new_pages = 0

        if no_new_pages >= 2:
            print("신규 게시글이 2페이지 연속 없어 종료")
            break

        # 다음 페이지 확인
        paging = result.get("paging", {})
        next_url = paging.get("next")

        if not next_url:
            print("다음 페이지가 없어 종료")
            break

        parsed = urlparse(next_url)
        query_params = parse_qs(parsed.query)

        # 액세스 토큰 유지
        query_params["access_token"] = [ACCESS_TOKEN]

        url = urlunparse(
            parsed._replace(
                query=urlencode(query_params, doseq=True)
            )
        )

        params = None

        time.sleep(2)

    except requests.exceptions.RequestException as e:
        print("요청 오류:", e)
        break

    except ValueError as e:
        print("응답 처리 오류:", e)
        break

# CSV 저장
df = pd.DataFrame(all_posts)

df.to_csv(
    "instagram_치이카와.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n====================")
print("인스타그램 수집 종료")
print("====================")
print(f"목표 수집량: {TARGET_COUNT}건")
print(f"실제 수집량: {len(df)}건")
print(f"요청 페이지: {page_num}개")
